<div style="background: linear-gradient(135deg, #1a1a2e 0%, #2d0a16 100%); padding: 28px 32px; border-radius: 12px; font-family: monospace;">
<h1 style="color: #FFD700; font-size: 2.2em; margin: 0; letter-spacing: 2px;">🌿 THE NEIGHBOR GAMES</h1>
<h2 style="color: #ffffff; font-size: 1.1em; margin: 8px 0 0 0; font-weight: normal;">📋 Instructor Scorecard</h2>
<p style="color: #aaaacc; margin: 8px 0 0 0; font-size: 0.85em;">Live leaderboard — auto-refreshes every 15 s, or click Refresh manually.</p>
</div>


### ⚙️ Configuration — must match the student notebook

In [ ]:
# Must match SCORES_FILE in student notebook
SCORES_FILE = '/home/jovyan/shared-readwrite/neighbor_games/scores.json'

ROUNDS = ['Tribute', 'Arena', 'Prize', 'Wildcard']
ROUND_META = {
    'Tribute' : {'emoji': '🌿', 'unit': 'K', 'label': 'RMSE'},
    'Arena'   : {'emoji': '⚔️', 'unit': 'K', 'label': 'RMSE'},
    'Prize'   : {'emoji': '🏆', 'unit': 'K', 'label': 'Avg Error'},
    'Wildcard': {'emoji': '🌟', 'unit': 'K', 'label': 'RMSE'},
}
MEDALS = ['🥇', '🥈', '🥉']
CHERRY = '#9E1B34'
GOLD   = '#FFD700'
print("Config loaded — reading from:", SCORES_FILE)

---
### Setup

In [ ]:
import json, os, time, threading, tempfile
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

print("Ready")

---
### Scorecard renderer — run once, then launch below

In [ ]:

def load_scores():
    try:
        with open(SCORES_FILE) as f:
            return json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        return []

def best_per_team(scores, round_name):
    bests = {}
    for s in scores:
        if s['round'] != round_name:
            continue
        t = s['team']
        if t not in bests or s['score'] < bests[t]['score']:
            bests[t] = s
    return sorted(bests.values(), key=lambda x: x['score'])

def score_bar(score, best, worst, color, width=200):
    rng = max(worst - best, 0.001)
    pct = max(5, int((1 - (score - best) / rng) * 100))
    return (
        f'<div style="background:#E8E4DE;border-radius:3px;height:6px;width:{width}px;">' +
        f'<div style="background:{color};width:{pct}%;height:100%;border-radius:3px;"></div></div>'
    )

def render_round_table(entries, meta, round_name):
    if not entries:
        return (
            f'<div style="color:#888;padding:12px;font-style:italic;font-size:13px;">' +
            f'No submissions yet for {round_name}.</div>'
        )
    best  = entries[0]["score"]
    worst = entries[-1]["score"]
    color = {"Tribute":"#6B6860","Arena":CHERRY,"Prize":GOLD,"Wildcard":"#2E7D50"}.get(round_name, CHERRY)
    rows = ""
    for i, e in enumerate(entries):
        medal = MEDALS[i] if i < 3 else f'<span style="color:#aaa;font-size:12px;">{i+1}</span>'
        bar   = score_bar(e["score"], best, worst, color)
        delta = "" if i == 0 else f'+{e["score"]-best:.2f}'
        extra = ""
        if round_name == "Arena" and "features" in e:
            extra = f'<span style="font-size:11px;color:#888;margin-left:6px;">features={e["features"]} k={e.get("k","?")}</span>'
        ts = e.get("timestamp","")
        rows += (
            f'<tr>' +
            f'<td style="padding:8px 10px;font-size:18px;text-align:center;">{medal}</td>' +
            f'<td style="padding:8px 4px;">' +
            f'<div style="font-size:14px;font-weight:600;color:#1a1a1a;">{e["team"]}{extra}</div>' +
            f'{bar}</td>' +
            f'<td style="padding:8px 10px;text-align:right;font-size:15px;font-weight:700;color:#1a1a1a;white-space:nowrap;">' +
            f'{e["score"]:.2f} <span style="font-size:11px;font-weight:400;color:#888;">{meta["unit"]}</span></td>' +
            f'<td style="padding:8px 10px;text-align:right;font-size:11px;color:#aaa;white-space:nowrap;">{delta}<br>{ts}</td>' +
            f'</tr>'
        )
    return (
        f'<table style="width:100%;border-collapse:collapse;font-family:system-ui,sans-serif;">' +
        f'<thead><tr style="border-bottom:2px solid #E0DDD6;">' +
        f'<th style="padding:6px 10px;width:36px;"></th>' +
        f'<th style="padding:6px 4px;text-align:left;font-size:12px;color:#888;font-weight:500;">Team</th>' +
        f'<th style="padding:6px 10px;text-align:right;font-size:12px;color:#888;font-weight:500;">{meta["label"]}</th>' +
        f'<th style="padding:6px 10px;text-align:right;font-size:12px;color:#888;font-weight:500;">Δ / Time</th>' +
        f'</tr></thead><tbody>{rows}</tbody></table>'
    )

def render_summary_strip(scores):
    parts = []
    for r in ROUNDS:
        m    = ROUND_META[r]
        ents = best_per_team(scores, r)
        n    = len(ents)
        best_str  = f'{ents[0]["score"]:.2f} {m["unit"]}' if ents else "—"
        leader    = ents[0]["team"] if ents else "—"
        parts.append(
            f'<div style="flex:1;min-width:130px;background:#F7F5F0;border-radius:10px;padding:12px;text-align:center;">' +
            f'<div style="font-size:1.4em;">{m["emoji"]}</div>' +
            f'<div style="font-size:13px;font-weight:600;color:#1a1a1a;margin:4px 0 2px;">{r}</div>' +
            f'<div style="font-size:11px;color:#888;">{n} team{"s" if n!=1 else ""} in</div>' +
            f'<div style="font-size:12px;color:#2E7D50;margin-top:4px;">{best_str}</div>' +
            f'<div style="font-size:11px;color:#555;overflow:hidden;text-overflow:ellipsis;white-space:nowrap;">{leader}</div>' +
            f'</div>'
        )
    return f'<div style="display:flex;gap:10px;flex-wrap:wrap;margin-bottom:20px;">{"".join(parts)}</div>'

def render_all(output_widget):
    scores  = load_scores()
    ts      = time.strftime("%H:%M:%S")
    n_teams = len({s["team"] for s in scores})
    parts   = [
        render_summary_strip(scores),
        f'<div style="font-size:11px;color:#aaa;margin-bottom:16px;">Last refreshed: {ts} · {n_teams} team(s) submitted</div>'
    ]
    for rnd in ROUNDS:
        meta    = ROUND_META[rnd]
        entries = best_per_team(scores, rnd)
        table   = render_round_table(entries, meta, rnd)
        parts.append(
            f'<div style="background:#fff;border:1px solid #E0DDD6;border-radius:10px;' +
            f'margin-bottom:16px;overflow:hidden;box-shadow:0 2px 8px rgba(0,0,0,0.06);">' +
            f'<div style="padding:10px 16px;border-bottom:1px solid #F0EDE8;' +
            f'display:flex;align-items:center;gap:8px;">' +
            f'<span style="font-size:1.2em;">{meta["emoji"]}</span>' +
            f'<span style="font-size:15px;font-weight:600;color:#1a1a1a;">{rnd} Round</span>' +
            f'<span style="font-size:11px;color:#aaa;margin-left:auto;">{len(entries)} submission{"s" if len(entries)!=1 else ""} · lower = better</span>' +
            f'</div>{table}</div>'
        )
    with output_widget:
        clear_output(wait=True)
        display(HTML("\n".join(parts)))

print("Renderer ready")


---
### 🚀 Launch Scorecard
Run this cell to display the live leaderboard.

In [ ]:

btn_refresh = widgets.Button(
    description="🔄 Refresh Now",
    layout=widgets.Layout(width="160px", height="36px"),
    style={"button_color": CHERRY, "font_weight": "bold"}
)
btn_toggle = widgets.Button(
    description="⏸ Pause Auto-Refresh",
    layout=widgets.Layout(width="210px", height="36px")
)
interval_slider = widgets.IntSlider(
    value=15, min=5, max=60, step=5,
    description="Interval (s):",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="320px")
)
status_lbl = widgets.Label(value="Auto-refresh: ON")
output     = widgets.Output()

_auto = [True]
_stop = threading.Event()

def do_refresh(_=None):
    render_all(output)

def toggle_auto(_):
    _auto[0] = not _auto[0]
    if _auto[0]:
        btn_toggle.description = "⏸ Pause Auto-Refresh"
        status_lbl.value = "Auto-refresh: ON"
    else:
        btn_toggle.description = "▶ Resume Auto-Refresh"
        status_lbl.value = "Auto-refresh: PAUSED"

btn_refresh.on_click(do_refresh)
btn_toggle.on_click(toggle_auto)

def _loop():
    while not _stop.is_set():
        secs = interval_slider.value
        for _ in range(secs):
            if _stop.is_set():
                return
            time.sleep(1)
        if _auto[0]:
            render_all(output)

_stop.clear()
threading.Thread(target=_loop, daemon=True).start()

controls = widgets.HBox(
    [btn_refresh, btn_toggle, interval_slider, status_lbl],
    layout=widgets.Layout(gap="12px", align_items="center", margin="0 0 12px 0")
)
display(controls, output)
do_refresh()
print("Scorecard live!")


---
### 🖊️ Manual Entry (fallback if a team can't submit from their notebook)

In [ ]:

def manual_submit(team, round_name, score, features=None, k=None):
    assert round_name in ROUNDS, f"round must be one of {ROUNDS}"
    entry = {"team": team, "round": round_name, "score": float(score),
             "timestamp": time.strftime("%H:%M:%S"), "manual": True}
    if features: entry["features"] = features
    if k:        entry["k"] = k
    try:
        with open(SCORES_FILE) as f:
            scores = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        scores = []
    scores = [s for s in scores if not (s["team"] == team and s["round"] == round_name)]
    scores.append(entry)
    dir_ = os.path.dirname(SCORES_FILE)
    os.makedirs(dir_, exist_ok=True)
    with tempfile.NamedTemporaryFile("w", dir=dir_, delete=False, suffix=".tmp") as f:
        json.dump(scores, f, indent=2)
        tmp = f.name
    os.replace(tmp, SCORES_FILE)
    print(f"Submitted: {team} · {round_name} · {score}")

# Example — uncomment and edit:
# manual_submit("Team Boiling Point Bandits", "Arena", 11.42, features=["MW","degree"], k=7)
# manual_submit("District 4", "Prize", 3.85)


---
### 🗑️ Reset (wipe all scores — use with caution)

In [ ]:
# import os
# if os.path.exists(SCORES_FILE):
#     os.remove(SCORES_FILE)
#     print("Scores cleared.")